# Day 86 — Architecture

This notebook covers Multi-Layer Perceptron (`MLP`) Architecture for Tabular Regression, Tapered Hidden Layers (`nn.Linear`, `nn.ReLU`), Regularization (`nn.Dropout`), Linear Output Head Design, and Parameter Counting
* **Time spent:** 2.0 hours

## Validation goals
- Design a tapered Multi-Layer Perceptron (`RetailDemandMLP`) tailored for continuous tabular retail demand regression
- Integrate `RetailDemandMLP` with the `build_dataloaders` pipeline and verify batch prediction shapes (`(32, 12)` to `(32, 1)`)
- Audit parameter counts (`2,945` total and trainable parameters) and per-layer tensor shapes via `named_parameters()`
- Analyze architectural trade-offs including linear output heads, model capacity on tabular datasets, and `nn.Dropout` behavior across `model.train()` vs. `model.eval()`

### 1. Designing a Tabular Regression Architecture

Tree ensembles like Random Forests split features along axis-aligned thresholds, whereas a Multi-Layer Perceptron (`MLP`) builds smooth, continuous combinations of features across stacked linear and non-linear layers.

For a tabular regression problem with `in_features` inputs and a single continuous output (`demand_units`):
- Input Projection (`nn.Linear(in_features, hidden_dim)`): Maps raw/preprocessed features into a higher-dimensional representation space where combinations of price, promo flags, and regional indicators can interact.
- Hidden Activations (`nn.ReLU()`): Introduces non-linearity between linear layers so the network can model curves and threshold effects rather than a single hyperplane.
- Tapered Hidden Layers (`hidden_dim` -> `hidden_dim` // 2): Gradually compresses representations toward the prediction target while keeping parameter counts modest for tabular datasets.
- Optional Regularization (`nn.Dropout`): Randomly zeroes a fraction of hidden activations during training (`model.train()`) to reduce co-adaptation; automatically disabled during evaluation (`model.eval()`).
- Linear Output Layer (`nn.Linear(..., 1)`): Outputs an unbounded real number suitable for continuous unit forecasting.

### 2. Implementing the Architecture (`day_86_architecture.py`)

Let's write a clean, reusable RetailDemandMLP module that builds on your `day_85_dataset.py` pipeline and provides parameter inspection and a shape check on a real batch.

In [1]:
import torch
import torch.nn as nn

from day_85_dataset import build_dataloaders


class RetailDemandMLP(nn.Module):
    """Multi-Layer Perceptron for tabular retail demand regression."""

    def __init__(
        self,
        in_features: int,
        hidden_dims: tuple[int, ...] = (64, 32),
        dropout: float = 0.1,
    ):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims must contain at least one hidden layer width.")

        layers: list[nn.Module] = []
        prev_dim = in_features

        for width in hidden_dims:
            layers.append(nn.Linear(prev_dim, width))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = width

        # Final linear head predicts continuous demand_units with no output activation
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def count_parameters(model: nn.Module) -> tuple[int, int]:
    """Returns (total_params, trainable_params)."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, _, num_features, source = build_dataloaders(batch_size=32)

    model = RetailDemandMLP(
        in_features=num_features,
        hidden_dims=(64, 32),
        dropout=0.1,
    ).to(device)

    batch_x, batch_y = next(iter(train_loader))
    batch_x = batch_x.to(device)

    with torch.no_grad():
        preds = model(batch_x)

    total_params, trainable_params = count_parameters(model)

    print("--- Day 86: RetailDemandMLP Architecture ---")
    print(f"Data source:       {source}")
    print(f"Device:            {device}")
    print(f"Input features:    {num_features}")
    print(f"Total params:      {total_params:,} (trainable: {trainable_params:,})")
    print(f"Batch X shape:     {tuple(batch_x.shape)}")
    print(f"Predictions shape: {tuple(preds.shape)}")
    print(f"Target y shape:    {tuple(batch_y.shape)}\n")
    print("Layer breakdown:")
    for name, param in model.named_parameters():
        print(f"  {name:28s} -> {tuple(param.shape)}")

--- Day 86: RetailDemandMLP Architecture ---
Data source:       synthetic_fallback
Device:            cpu
Input features:    12
Total params:      2,945 (trainable: 2,945)
Batch X shape:     (32, 12)
Predictions shape: (32, 1)
Target y shape:    (32, 1)

Layer breakdown:
  network.0.weight             -> (64, 12)
  network.0.bias               -> (64,)
  network.3.weight             -> (32, 64)
  network.3.bias               -> (32,)
  network.6.weight             -> (1, 32)
  network.6.bias               -> (1,)


### 3. Architectural Trade-Offs on Tabular Data

- Width & Depth: Tabular datasets with dozens of features rarely need massive networks. A compact `64 -> 32 -> 1` architecture trains quickly, avoids severe overfitting on moderate row counts, and runs easily on CPU.
- Dropout Placement: Applying light dropout (`0.1`) inside hidden blocks helps prevent memorization, while keeping it low (or `0.0` on small/clean datasets) avoids underfitting when signal is already structured.
- Output Head: Keeping the output layer linear avoids clipping predictions at zero inside the layer itself—if targets are non-negative, you can inspect or clamp/post-process predictions cleanly during evaluation or domain logic rather than letting `ReLU` create zero-gradient dead outputs at initialization.

### Guided Question:

Why does a tabular regression network use `nn.ReLU()` between hidden `nn.Linear` layers, but leave the final `nn.Linear(prev_dim, 1)` layer without a hidden activation?

### Answer:

A tabular regression network uses `nn.ReLU()` between hidden layers to learn nonlinear relationships, but leaves the final `nn.Linear(prev_dim, 1)` layer raw so the model can output any real number without artificial clipping or range limits.

Why `nn.ReLU()` Belongs in Hidden Layers
- Enables nonlinear combinations: Stacking linear layers without a nonlinearity collapses the entire network into a single linear transformation ($\mathbf{W}_k \cdots \mathbf{W}_2 \mathbf{W}_1 \mathbf{x} + \mathbf{b}$). `nn.ReLU()` introduces piecewise-linear bends so the network can model interactions and curves across tabular features.
- Keeps gradients flowing in active units: For positive pre-activations, `nn.ReLU()` has a clean derivative of $1$, avoiding the gradient shrinkage of sigmoid or tanh in deeper networks.
- Sparsifies hidden activations: Zeroing negative activations creates sparse representations that are fast to compute and easy to optimize on tabular data.

**Why the Final Regression Layer Has No Activation**

- Unbounded output range: Standard regression targets (such as residuals, prices, temperatures, or scaled demand) can take negative, zero, or arbitrarily large positive values. A linear output layer $\mathbf{w}^\top \mathbf{h} + b$ leaves the full real range open.
- Avoids dead or saturated outputs: Putting `nn.ReLU()` on the final layer would clamp every prediction less than or equal to $0$ to exact 0.0 with zero gradient whenever pre-activation is negative—harmful if targets can be negative or if early predictions start negative.
- Preserves loss-function behavior: Regression losses like `nn.MSELoss`, `nn.SmoothL1Loss`, and nn.L1Loss expect raw continuous predictions and penalize deviation directly without needing a classification squashing function like sigmoid or softmax.

**When You Might Add a Final Activation Anyway**

Known domain bounds: Only add an output activation (or transform targets instead) when domain rules strictly require a bound—for example, `nn.Softplus()` or `torch.exp()` if a target must be strictly positive, or `nn.Sigmoid()` if scaled strictly to $(0, 1)$. Even then, many practitioners train in an unconstrained space (such as predicting $\log(y)$) and invert the transform afterward rather than squashing inside the final layer.

### Guided Question:

On a tabular dataset with `~12–20` features and a few thousand rows, what risks arise if you make an `MLP` excessively deep or wide without regularization?

### Answer:

On a tabular dataset with roughly 12–20 features and a few thousand rows, an overly deep or wide MLP lacking regularization has high parameter capacity relative to sample count, leading to overfitting, noisy predictions, and unstable training.

**1. Severe Overfitting and Memorization**

- Fitting training noise: Thousands of parameters can memorize idiosyncratic quirks, outliers, or label noise in a few thousand rows rather than learning generalizable feature relationships.
- Wide gap between train and validation error: Training loss may drop near zero while validation or test error rises or plateaus early, hurting performance on unseen data.

**2. Poor Extrapolation and Jagged Decision Surfaces**

- Piecewise-linear fragmentation: With activations such as `nn.ReLU()`, excessive width and depth partition input space into many small regions with sharp bends, producing erratic predictions between sparse training points.
- Unreliable extrapolation: Outside the observed feature ranges (such as unusually high prices, promotions, or demand spikes), unregularized MLPs often extrapolate wildly instead of following smooth trends.

**3. Optimization Instability and Wasted Compute**

- Harder gradient flow: Deep networks without normalization, skip connections, or careful initialization can suffer vanishing or exploding gradients, making learning-rate tuning sensitive.
- Noisy mini-batch updates: High-capacity models can fluctuate across epochs on small-to-moderate datasets, slowing convergence and wasting CPU/GPU time compared to a compact MLP or gradient-boosted trees.

**Practical Guardrails for This Data Regime**

- Keep capacity modest: Start with 1–3 hidden layers and moderate widths (such as 32–128 units) before scaling up.
- Add regularization: Use weight decay (L2 penalty via optimizer), dropout (nn.Dropout), early stopping on validation loss, and standardized numerical features.
- Compare against simpler baselines: Benchmark against ridge/linear models and gradient-boosted decision trees, which often match or beat MLPs on small-to-medium tabular data.

### Guided Question:

Why does `nn.Dropout` behave differently when `model.train()` is active compared to `model.eval()`?

### Answer:

`nn.Dropout` behaves differently between `model.train()` and `model.eval()` because training randomly zeroes activations to prevent co-adaptation, while evaluation uses all active units without random dropping so predictions stay deterministic and properly scaled.

**Behavior During `model.train()`**

- Randomly zeroes activations: Each element in the input tensor is independently set to 0 with probability p, forcing remaining units to carry predictive signal on their own instead of relying on specific neighbor combinations.
- Applies inverted dropout scaling: Remaining non-zero activations are divided by $1 - p$ during training. This keeps expected activation magnitudes consistent with full-network inference without changing weights at test time.
- Changes every forward pass: Because a fresh random mask is drawn per pass, two forward calls on the same input during training can produce different outputs.

**Behavior During `model.eval()`**

- Acts as an identity pass-through: `nn.Dropout` disables random masking and returns the input tensor unchanged `(x).Deterministic outputs`: Passing the same input multiple times during validation or testing yields identical predictions, making evaluation repeatable and reliable.Preserves expected scale: Because training already scaled surviving activations by $\frac{1}{1 - p}$, evaluation needs no extra scaling factor—every unit contributes at its normal scale.

**Why Switching Modes Matters**

- Forgetting `model.eval()`: Leaving dropout active during inference adds random noise to predictions and makes metrics fluctuate across runs.
- Forgetting `model.train()`: Leaving dropout off during training removes regularization and can lead to faster overfitting on smaller datasets.